#      RAG SYSTEM

### document loading

In [75]:

import pathlib

def load_documents(directory_path):
    docs=[]
    path = pathlib.Path(directory_path)
    
    if not path.exists() or not path.is_dir():
        raise ValueError(f"Invalid directory path: {directory_path}")
    
    for file_path in path.iterdir():
        if file_path.suffix in ['.txt', '.md']:
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                    if not content.strip():
                        print(f"Warning: {file_path} is empty.")
                    metadata = {
                        'file_name': file_path.name,
                        "source": str(file_path),
                    }
                    docs.append({"content": content, "metadata": metadata})
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return docs

loaded_docs = load_documents(r"C:\AI_PROJECTS\ice_pytorch\data")
loaded_docs[:2]


[{'content': '# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI\'s `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, but geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in high-dimensional spaces. \n\nAs the number of dimensions increases, the volume of the space increases so rapidly that the available data becomes sparse. More importantly for retrieval systems, the concept of "distance" becomes less meanin

### chunking

In [76]:
import re
import bisect

def get_sections_with_hierarchy(content):
    # 1. Clean the content of windows style carriage returns just in case
    cleaned_content = content.replace('\r\n', '\n')
    
    # 2. Resilient pattern: Looks for 1 to 6 '#' signs at the start of any line
    # followed by at least one space or tab, capturing the text until the end of that line.
    header_pattern = re.compile(r'^(#{1,6})[ \t]+(.*)$', re.MULTILINE)
    
    sections = []
    hierarchy_stack = []

    for match in header_pattern.finditer(cleaned_content):
        # match.group(1) is the actual hashtags string (e.g. "##")
        level = len(match.group(1))        
        header_text = match.group(2).strip() 
        start_char = match.start()
        
        # Maintain the stack hierarchy
        while hierarchy_stack and hierarchy_stack[-1][0] >= level:
            hierarchy_stack.pop()
            
        hierarchy_stack.append((level, header_text))
        hierarchy_path = " > ".join([h[1] for h in hierarchy_stack])
        
        sections.append({
            "start_char": start_char,
            "section_path": hierarchy_path
        })
        
    return sections
'''
header_data=[]
for i in loaded_docs:
    header_data.append(get_sections_with_hierarchy(i["content"]))
    '''

def fixed_size_chunking(documents,n=0, chunk_size=512):
    chunks=[]
    content=documents[n]["content"]
    file_name=documents[n]["metadata"]["file_name"]
    header_info=get_sections_with_hierarchy(content)
    header_starts=[h["start_char"] for h in header_info]
    for i in range(0, len(content), chunk_size):
        chunk=content[i:i+chunk_size]

        if chunk:
            chunks.append({"content": chunk,
                          "metadata": {
                                "file_name": file_name,
                                "chunking_strategy": "fixed",
                                "chunk_index": i//chunk_size,
                                "start_char": i,
                                "section": header_info[bisect.bisect_right(header_starts, i) - 1]["section_path"] if header_info and bisect.bisect_right(header_starts, i) > 0 else "",
                                "words": len(chunk.split()),
                                "lines": len(chunk.splitlines())
                          }}
                          )
    return chunks

def sliding_window_chunking(documents,n=0, chunk_size=512, overlap=128):
    chunks=[]
    file_name=documents[n]["metadata"]["file_name"]
    step=chunk_size-overlap
    header_info=get_sections_with_hierarchy(documents[n]["content"])
    header_starts=[h["start_char"] for h in header_info]
    for i in range(0, len(documents[n]["content"]), step):
        chunk=documents[n]["content"][i:i+chunk_size]
        if chunk:
            chunks.append({"content": chunk,
                          "metadata": {
                                "file_name": file_name,
                                "chunking_strategy": "sliding",
                                "chunk_index": i//chunk_size,
                                "section": header_info[bisect.bisect_right(header_starts, i) - 1]["section_path"] if header_info and bisect.bisect_right(header_starts, i) > 0 else "",
                                "start_char": i,
                                "end_char": i+len(chunk),
                                "characters": len(chunk),
                                "words": len(chunk.split()),
                                "lines": len(chunk.splitlines())
                          }}
                          )
    return chunks

def header_aware_chunking(documents):
    chunks=[]
    for i, doc in enumerate(documents):
        sections = re.split(r'\n(?=#)', doc["content"])
        file_name=doc["metadata"]["file_name"]
        header_info=get_sections_with_hierarchy(doc["content"])
        header_starts=[h["start_char"] for h in header_info]
        current_char_pos = 0
        for idx, section in enumerate(sections):
            if section.strip():
                section_start = doc["content"].find(section, current_char_pos)
                pos = bisect.bisect_right(header_starts, section_start)
                section_path = (
                    header_info[pos - 1]["section_path"]
                    if header_info and pos > 0
                    else ""
                )
                clean_content = re.sub(r'^\s*#+.*(?:\n|$)', '', section).strip()
                chunks.append({
                    "content": clean_content,
                    "metadata": {
                        "file_name": file_name,
                        "chunking_strategy": "header_aware",
                        "chunk_index": idx,
                        "section": section_path,
                        "characters": len(clean_content),
                        "words": len(clean_content.split()),
                        "lines": len(clean_content.splitlines())
                    }
                })
                current_char_pos = section_start + len(section)
    return chunks

In [77]:
for i in loaded_docs:
    print(get_sections_with_hierarchy(i["content"]))

[{'start_char': 0, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)'}, {'start_char': 66, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 1. The Geometry of High-Dimensional Space'}, {'start_char': 593, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 1. The Geometry of High-Dimensional Space > 1.1 The Curse of Dimensionality'}, {'start_char': 1372, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics'}, {'start_char': 1495, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.1 Euclidean Distance (L2)'}, {'start_char': 1736, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.2 Cosine Similarity'}, {'start_char': 2196, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.3 Inner Prod

In [78]:
def preprocess_document(documents, n=0,chunking_strategy='fixed', chunk_size=512, overlap=128):
    if chunking_strategy == 'fixed':
        return fixed_size_chunking(documents, n, chunk_size)
    elif chunking_strategy == 'sliding':
        return sliding_window_chunking(documents, n, chunk_size, overlap)
    elif chunking_strategy == 'header_aware':
        return header_aware_chunking(documents)
    else:
        raise ValueError(f"Unknown chunking strategy: {chunking_strategy}")

In [79]:
def preprocess_documents(documents, chunking_strategy='fixed', chunk_size=512, overlap=128):
    all_chunks=[]
    for n in range(len(documents)):
        
        if(chunking_strategy == "header_aware"):
            chunks=preprocess_document(documents, n, chunking_strategy, chunk_size, overlap)
            all_chunks.extend(chunks)
            break
        
        chunks=preprocess_document(documents, n, chunking_strategy, chunk_size, overlap)
        all_chunks.extend(chunks)
    print(f"Total chunks created using {chunking_strategy}: {len(all_chunks)}")
    return all_chunks

In [80]:
processed_docs = preprocess_documents(loaded_docs, chunking_strategy='fixed', chunk_size=512)
 # Print first 100 characters of each chunk
 

Total chunks created using fixed: 51


In [81]:
print(f"Total documents loaded: {len(loaded_docs)}")
print(f"Total chunks created: {len(processed_docs)}")
for i, chunk in enumerate(processed_docs):
    print(f"\nChunk {i+1}:\n{chunk['content'][:10]}...") 

Total documents loaded: 5
Total chunks created: 51

Chunk 1:
# Advanced...

Chunk 2:
t geometry...

Chunk 3:
between th...

Chunk 4:
n distance...

Chunk 5:
ed with wo...

Chunk 6:
rest Neigh...

Chunk 7:
ains in sp...

Chunk 8:
eaviate.

...

Chunk 9:
ps.

### 4...

Chunk 10:
d entry po...

Chunk 11:
imum on La...

Chunk 12:
dataset to...

Chunk 13:
 the syste...

Chunk 14:
le, a 1024...

Chunk 15:
chine.

--...

Chunk 16:
oat32')

#...

Chunk 17:
5 nearest ...

Chunk 18:
-NN finds ...

Chunk 19:
ply metada...

Chunk 20:
 scan.

##...

Chunk 21:
# Neural N...

Chunk 22:
buted to t...

Chunk 23:
update ste...

Chunk 24:
calability...

Chunk 25:
commonly u...

Chunk 26:
ecution un...

Chunk 27:
nse numeri...

Chunk 28:
ine:

1. Q...

Chunk 29:
imilarity ...

Chunk 30:
quality ev...

Chunk 31:
# Building...

Chunk 32:
retrieval ...

Chunk 33:
becomes im...

Chunk 34:
the previo...

Chunk 35:
uire objec...

Chunk 36:
 but also ...

Chunk 37:
nk boundar...

Chunk 38:
guage mode...

Chunk 3

In [82]:
loaded_docs

[{'content': '# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI\'s `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, but geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in high-dimensional spaces. \n\nAs the number of dimensions increases, the volume of the space increases so rapidly that the available data becomes sparse. More importantly for retrieval systems, the concept of "distance" becomes less meanin

In [83]:
print(type(processed_docs))
print(type(loaded_docs))

<class 'list'>
<class 'list'>


In [84]:
c=0
for i in processed_docs:
    print(f"Chunk {c}: {i} \n")
    c += 1

Chunk 0: {'content': "# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI's `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, bu", 'metadata': {'file_name': 'advanced_vector_search.md', 'chunking_strategy': 'fixed', 'chunk_index': 0, 'start_char': 0, 'section': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)', 'words': 67, 'lines': 7}} 

Chunk 1: {'content': 't geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and 

In [85]:
from fastembed import TextEmbedding
import numpy as np

def get_similarity(v1,v2):
    return np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))



In [86]:
model=TextEmbedding()
def model_embedding(processed_docs):
    model=TextEmbedding()
    embeddings=list(model.embed([chunk['content'] for chunk in processed_docs ]))
    return embeddings

In [87]:

embeddings=model_embedding(processed_docs)
embeddings

[array([-4.49920520e-02, -2.73703858e-02, -1.44381430e-02,  2.44774725e-02,
         2.53360998e-02, -2.20865272e-02, -6.30892664e-02,  4.16632220e-02,
         3.09898295e-02,  3.08577325e-02,  8.11072253e-03, -7.78840706e-02,
         2.04617418e-02,  3.36053371e-02,  4.21916097e-02,  2.93254126e-02,
        -5.78252226e-03,  5.18082306e-02, -7.57573172e-03, -2.73175463e-02,
         7.17547983e-02,  9.11465567e-03, -3.52169164e-02, -2.37245243e-02,
        -8.69194698e-03,  2.71326117e-02, -1.80575848e-02, -4.14782874e-02,
        -3.49527225e-02, -2.23612875e-01,  2.94575095e-02, -7.37758726e-03,
         1.08530447e-01,  2.33282335e-02, -3.60623337e-02,  3.11483443e-02,
         1.23840431e-02,  2.30508316e-02, -6.75343152e-04, -3.82551327e-02,
        -3.52433361e-02,  1.74499415e-02, -2.99594756e-02, -2.33678631e-02,
        -1.61289778e-02, -1.25953974e-02, -5.53748347e-02, -6.20192848e-03,
        -6.39875233e-02,  2.40415558e-02, -3.97280091e-03, -1.85595527e-02,
        -2.9

In [88]:
print(f"Semantic matching test (Chunks 0 & 1): {get_similarity(embeddings[0], embeddings[1])}")
print(f"Semantic matching test (Chunks 0 & 2): {get_similarity(embeddings[0], embeddings[2])}")
print(f"Semantic matching test (Chunks 1 & 2): {get_similarity(embeddings[1], embeddings[2])}")

Semantic matching test (Chunks 0 & 1): 0.6800877451896667
Semantic matching test (Chunks 0 & 2): 0.6792548894882202
Semantic matching test (Chunks 1 & 2): 0.6598607301712036


In [89]:
sliding_window_chunks = preprocess_documents(loaded_docs, chunking_strategy='sliding', chunk_size=512, overlap=128)
header_aware_chunks = preprocess_documents(loaded_docs, chunking_strategy='header_aware', chunk_size=512, overlap=128)

# header_aware_chunks

Total chunks created using sliding: 69
Total chunks created using header_aware: 82


In [90]:


sliding_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)


In [91]:
def similarity_comparison(query, embeddings):
    query_embedding=list(model.embed([query]))[0]
    for i, emb in enumerate(embeddings):
        sim=get_similarity(query_embedding, emb)
        print(f"Similarity between query and chunk {i}: {sim}")

In [92]:
def similarity_comparison_display_all(query):
    print(f"\nQuery: {query}\n")
    print("Similarity with fixed size chunks:")
    similarity_comparison(query,embeddings)
    print("\nSimilarity with sliding window chunks:")
    similarity_comparison(query,sliding_embeddings)
    print("\nSimilarity with header aware chunks:")
    similarity_comparison(query,header_aware_embeddings)

In [93]:
query1="what is stochastic gradient descent?"
query2="explain the concept of overfitting in machine learning"
query3="what is cosine similarity?"


In [94]:
similarity_comparison_display_all(query1)
similarity_comparison_display_all(query2)
similarity_comparison_display_all(query3)


Query: what is stochastic gradient descent?

Similarity with fixed size chunks:
Similarity between query and chunk 0: 0.6019666194915771
Similarity between query and chunk 1: 0.5226749777793884
Similarity between query and chunk 2: 0.5221571326255798
Similarity between query and chunk 3: 0.4609302878379822
Similarity between query and chunk 4: 0.5179353952407837
Similarity between query and chunk 5: 0.5752649903297424
Similarity between query and chunk 6: 0.5423742532730103
Similarity between query and chunk 7: 0.4802510440349579
Similarity between query and chunk 8: 0.5507391095161438
Similarity between query and chunk 9: 0.6433355212211609
Similarity between query and chunk 10: 0.5712867379188538
Similarity between query and chunk 11: 0.5421537756919861
Similarity between query and chunk 12: 0.5706848502159119
Similarity between query and chunk 13: 0.5690937638282776
Similarity between query and chunk 14: 0.5470132231712341
Similarity between query and chunk 15: 0.5809239149093628
S

In [95]:
import numpy as np

def get_top_five_similarities_fast(query, embedding_matrix, model):
    # 1. Get the query embedding as a 1D numpy array
    query_vector = np.array(list(model.embed([query]))[0])
    
    # 2. Compute similarity for ALL chunks at once using vector dot product
    # Assumes embedding_matrix has shape (num_chunks, embedding_dim)
    # and vectors are normalized. If not normalized, divide by norms.
    scores = np.dot(embedding_matrix, query_vector)
    
    # 3. Use argsort to get indices of the highest scores
    # np.argsort returns indices from smallest to largest; [::-1] reverses it
    top_indices = np.argsort(scores)[::-1][:5]
    
    # 4. Return as a list of (index, score) tuples to match your original format
    return [(idx, float(scores[idx])) for idx in top_indices]

def similarity_comparison_top_result(query):
    query_embedding=list(model.embed([query]))[0]
    
    def get_top_similarity(embeddings):
        max_sim=-1
        top_index=-1
        for i, emb in enumerate(embeddings):
            sim=get_similarity(query_embedding, emb)
            if sim > max_sim:
                max_sim=sim
                top_index=i
        return top_index, max_sim
    
    top_fixed_index, top_fixed_sim=get_top_similarity(embeddings)
    top_sliding_index, top_sliding_sim=get_top_similarity(sliding_embeddings)
    top_header_index, top_header_sim=get_top_similarity(header_aware_embeddings)
    
    return {
        "fixed": {"index": top_fixed_index, "similarity": top_fixed_sim},
        "sliding": {"index": top_sliding_index, "similarity": top_sliding_sim},
        "header": {"index": top_header_index, "similarity": top_header_sim}
    }

def similarity_comparison_top_result_display(query):
    result = similarity_comparison_top_result(query)
    print(f"\nQuery: {query}\n")
    print(f"Top similarity with fixed size chunks: Chunk {result['fixed']['index']} (Similarity: {result['fixed']['similarity']})")
    print(f"Top similarity with sliding window chunks: Chunk {result['sliding']['index']} (Similarity: {result['sliding']['similarity']})")
    print(f"Top similarity with header aware chunks: Chunk {result['header']['index']} (Similarity: {result['header']['similarity']})")
    

def retrieve_top_chunks(query, embedding_matrix, model, processed_docs, k=5):
    chunk_indices = get_top_five_similarities_fast(query, embedding_matrix, model)
    top_chunks = []
    for idx, score in chunk_indices:
        if idx < len(processed_docs):  # Ensure the index is within bounds
            top_chunks.append({
                "chunk_index": idx,
                "similarity_score": score,
                 "section": processed_docs[idx]["metadata"]["section"] if idx < len(processed_docs) and "metadata" in processed_docs[idx] else "N/A",
                "chunk_content": processed_docs[idx]["content"]  # Assuming each chunk is a list of dicts
            })
    return top_chunks

In [96]:
similarity_comparison_top_result(query1)
similarity_comparison_top_result(query2)
similarity_comparison_top_result(query3)

{'fixed': {'index': 4, 'similarity': np.float32(0.79804)},
 'sliding': {'index': 37, 'similarity': np.float32(0.8376154)},
 'header': {'index': 5, 'similarity': np.float32(0.8927598)}}

In [97]:
queries=["What is SGD?",
"What is mini-batch SGD?",
"What is a GPU?",
"What is an NPU?",
"What are embeddings?",
"What is vector space?",
"What is cosine similarity?",
"What is precision?",
"What is recall?",
"What is F1 score?"]

In [98]:

for q in queries:
    similarity_comparison_top_result_display(q)


Query: What is SGD?

Top similarity with fixed size chunks: Chunk 22 (Similarity: 0.6167911291122437)
Top similarity with sliding window chunks: Chunk 30 (Similarity: 0.6622854471206665)
Top similarity with header aware chunks: Chunk 35 (Similarity: 0.6910408735275269)

Query: What is mini-batch SGD?

Top similarity with fixed size chunks: Chunk 23 (Similarity: 0.7869912385940552)
Top similarity with sliding window chunks: Chunk 31 (Similarity: 0.7869912385940552)
Top similarity with header aware chunks: Chunk 36 (Similarity: 0.7840518951416016)

Query: What is a GPU?

Top similarity with fixed size chunks: Chunk 24 (Similarity: 0.6641398668289185)
Top similarity with sliding window chunks: Chunk 33 (Similarity: 0.6972606778144836)
Top similarity with header aware chunks: Chunk 39 (Similarity: 0.7011777758598328)

Query: What is an NPU?

Top similarity with fixed size chunks: Chunk 25 (Similarity: 0.6473544239997864)
Top similarity with sliding window chunks: Chunk 34 (Similarity: 0.6

In [99]:
# 1. Semantic Ambiguity Tests
semantic_ambiguity_queries = [
    "What are precision, recall, and F1 score, and why are they insufficient for evaluating a modern generative pipeline?",
    "Explain the concept of embeddings. How does a static embedding model differ from how embeddings are handled right before a Transformer's attention layer?",
    "What is the difference between a position bias in an LLM judge and positional encoding?"
]

# 2. Deep Context & Hierarchy Tests
deep_context_queries = [
    "Why is exact k-NN search impossible at a massive scale, and how do Voronoi cells in an Inverted File Index solve this?",
    "Explain the difference between factual hallucinations and faithfulness hallucinations.",
    "What happens to the gradient when the learning rate is too large versus too small?"
]

# 3. Code & Math Extraction Tests
code_math_queries = [
    "Show me the Python implementation for scaled dot-product attention and explain what the d_k variable represents.",
    "I need to build an IVF-PQ index. Provide the FAISS Python code and explain how the compression ratio is achieved.",
    "Provide the mathematical formula for Cosine Similarity and show the Python code to compute it."
]

# 4. Advanced Cross-Document Synthesis Tests
cross_document_synthesis_queries = [
    "Trace the evolution of search: How does the synonym problem in BM25 relate to the curse of dimensionality in high-dimensional vector spaces?",
    "If I am using an HNSW index, why might applying a pre-filtering metadata constraint completely destroy my search process?",
    "How do parallel execution units in GPUs specifically accelerate the scaled dot-product calculations inside a multi-head attention mechanism?"
]


In [100]:
all_experimental_queries= (
    semantic_ambiguity_queries + 
    deep_context_queries + 
    code_math_queries + 
    cross_document_synthesis_queries
)

## TESTING BRO

In [101]:
model=TextEmbedding()
print(processed_docs)
print(processed_docs[0]["content"])
embeddings=model_embedding(processed_docs)


[{'content': "# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI's `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, bu", 'metadata': {'file_name': 'advanced_vector_search.md', 'chunking_strategy': 'fixed', 'chunk_index': 0, 'start_char': 0, 'section': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)', 'words': 67, 'lines': 7}}, {'content': 't geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in

In [102]:
print(f"semantic_ambiguity_queries: {semantic_ambiguity_queries}")
for i in semantic_ambiguity_queries:
    similarity_comparison_top_result(i)
print(f"deep_context_queries: {deep_context_queries}")
for i in deep_context_queries:
    similarity_comparison_top_result(i)
print(f"code_math_queries: {code_math_queries}")
for i in code_math_queries:
    similarity_comparison_top_result(i)
print(f"cross_document_synthesis_queries: {cross_document_synthesis_queries}")
for i in cross_document_synthesis_queries:
    similarity_comparison_top_result(i)

semantic_ambiguity_queries: ['What are precision, recall, and F1 score, and why are they insufficient for evaluating a modern generative pipeline?', "Explain the concept of embeddings. How does a static embedding model differ from how embeddings are handled right before a Transformer's attention layer?", 'What is the difference between a position bias in an LLM judge and positional encoding?']
deep_context_queries: ['Why is exact k-NN search impossible at a massive scale, and how do Voronoi cells in an Inverted File Index solve this?', 'Explain the difference between factual hallucinations and faithfulness hallucinations.', 'What happens to the gradient when the learning rate is too large versus too small?']
code_math_queries: ['Show me the Python implementation for scaled dot-product attention and explain what the d_k variable represents.', 'I need to build an IVF-PQ index. Provide the FAISS Python code and explain how the compression ratio is achieved.', 'Provide the mathematical for

In [103]:
def qualitative_reranker(query, retrieved_chunks):
    boosted_chunks = []
    query_words = set(query.lower().split())
    
    for chunk in retrieved_chunks:
        score = chunk["similarity_score"]
        section_path = chunk["metadata"]["section"].lower()
        
        # Qualitative Rule: If terms from the hierarchy path are explicitly 
        # typed by the user, boost the relevance score by 15%
        path_words = set(section_path.replace(" > ", " ").split())
        matches = query_words.intersection(path_words)
        
        if matches:
            score += 0.15 * len(matches) # Add a qualitative weight bonus
            
        chunk["reranked_score"] = score
        boosted_chunks.append(chunk)
        
    # Sort by the new score
    return sorted(boosted_chunks, key=lambda x: x["reranked_score"], reverse=True)



In [104]:
a=similarity_comparison_top_result("What is the difference between a position bias in an LLM judge and positional encoding?")
print(f"Top chunk before reranking: {a}")

Top chunk before reranking: {'fixed': {'index': 44, 'similarity': np.float32(0.74809706)}, 'sliding': {'index': 61, 'similarity': np.float32(0.7526419)}, 'header': {'index': 75, 'similarity': np.float32(0.79203016)}}


In [105]:
retrieve_top_chunks("what is similarity?", embeddings,model,processed_docs, k=5)

[{'chunk_index': np.int64(4),
  'similarity_score': 0.7021068334579468,
  'section': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.2 Cosine Similarity',
  'chunk_content': 'ed with word count or token density) is less important than its semantic direction.\n\n$$similarity = \\cos(\\theta) = \\frac{A \\cdot B}{||A|| ||B||}$$\n\n### 2.3 Inner Product\n\nThe inner product (or dot product) is an unnormalized version of cosine similarity. If all vectors in the database are normalized to have a length of 1 (L2 normalization), the inner product is mathematically equivalent to cosine similarity but computationally faster to execute.\n\n---\n\n## 3. Exact Search vs. Approximate Search\n\n### 3.1 k-Nea'},
 {'chunk_index': np.int64(27),
  'similarity_score': 0.6817388534545898,
  'section': 'Neural Networks in Modern Deep Learning > Retrieval Systems',
  'chunk_content': 'ine:\n\n1. Query\n2. Embedding\n3. Similarity Search\n4. Ranking\n5. Context Const

In [106]:
retrieve_top_chunks("what is deep learning?", embeddings, model, processed_docs, k=5)

[{'chunk_index': np.int64(20),
  'similarity_score': 0.762347936630249,
  'section': 'Neural Networks in Modern Deep Learning',
  'chunk_content': '# Neural Networks in Modern Deep Learning\n\n## Introduction\n\nDeep learning systems are built from layers of interconnected mathematical operations. These operations allow models to learn representations directly from data instead of relying on manually engineered features.\n\nA neural network consists of neurons organized into layers. The most common architecture is the feedforward neural network, where information moves from input to output without cycles.\n\nThe success of modern deep learning is largely attri'},
 {'chunk_index': np.int64(24),
  'similarity_score': 0.6586467027664185,
  'section': 'Neural Networks in Modern Deep Learning > Mini-Batch Gradient Descent',
  'chunk_content': 'commonly uses mini-batches because they efficiently utilize parallel computation resources.\n\nThe term "parallel computation resources" will become 

In [107]:
retrieve_top_chunks("what is recall?", embeddings, model,processed_docs, k=5)

[{'chunk_index': np.int64(34),
  'similarity_score': 0.5875473022460938,
  'section': 'Building Modern Retrieval Systems > Retrieval Evaluation',
  'chunk_content': 'uire objective evaluation.\n\nSeveral metrics are commonly used.\n\n### Precision\n\nPrecision measures the fraction of retrieved documents that are relevant.\n\nHigh precision means fewer irrelevant results.\n\n### Recall\n\nRecall measures the fraction of relevant documents successfully retrieved.\n\nHigh recall means fewer missed documents.\n\n### F1 Score\n\nF1 Score combines precision and recall.\n\nPoor chunking can reduce all three metrics simultaneously.\n\nNotice that retrieval quality now depends not only on embeddings'},
 {'chunk_index': np.int64(17),
  'similarity_score': 0.5836571455001831,
  'section': 'Perform a search > 8. Evaluation Metrics for Vector Search > 8.1 Recall@K',
  'chunk_content': '-NN finds items [A, B, C, D, E], and HNSW returns [A, B, F, D, G], the Recall@5 is 60%.\n\n### 8.2 Queries Per Se

In [108]:
print(len(processed_docs))
print(len(embeddings))

51
51


In [109]:
print("Retrieving top chunks for 'what is GPU?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", header_aware_embeddings, model, header_aware_chunks, k=5))

Retrieving top chunks for 'what is GPU?':


using fixed size chunking embeddings:
[{'chunk_index': np.int64(24), 'similarity_score': 0.6607635021209717, 'section': 'Neural Networks in Modern Deep Learning > Mini-Batch Gradient Descent', 'chunk_content': 'commonly uses mini-batches because they efficiently utilize parallel computation resources.\n\nThe term "parallel computation resources" will become important when discussing modern accelerators.\n\n---\n\n## Hardware Acceleration\n\nTraining deep neural networks requires massive numbers of matrix multiplications.\n\nHardware accelerators include:\n\n### CPU\n\nCentral Processing Units are flexible but generally slower for large-scale tensor operations.\n\n### GPU\n\nGraphics Processing Units provide thousands of parallel ex'}, {'chunk_index': np.int64(25), 'similarity_score': 0.6558100581169128, 'section': 'Neural Networks in Modern Deep Learning > Hardware Acceleration > GPU', 'chunk_content': 'ecution units.\n\nThese parallel execut

In [110]:
print("Retrieving top chunks for 'why do mini-batches work well on GPUs?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", header_aware_embeddings, model, header_aware_chunks, k=5))

Retrieving top chunks for 'why do mini-batches work well on GPUs?':


using fixed size chunking embeddings:
[{'chunk_index': np.int64(24), 'similarity_score': 0.8285914659500122, 'section': 'Neural Networks in Modern Deep Learning > Mini-Batch Gradient Descent', 'chunk_content': 'commonly uses mini-batches because they efficiently utilize parallel computation resources.\n\nThe term "parallel computation resources" will become important when discussing modern accelerators.\n\n---\n\n## Hardware Acceleration\n\nTraining deep neural networks requires massive numbers of matrix multiplications.\n\nHardware accelerators include:\n\n### CPU\n\nCentral Processing Units are flexible but generally slower for large-scale tensor operations.\n\n### GPU\n\nGraphics Processing Units provide thousands of parallel ex'}, {'chunk_index': np.int64(23), 'similarity_score': 0.7696696519851685, 'section': 'Neural Networks in Modern Deep Learning > Stochastic Gradient Descent', 'chunk_content': 'calability\n\

In [111]:
from collections import Counter

contents = [chunk["content"] for chunk in header_aware_chunks]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in header aware: {len(duplicates)}")

contents = [chunk["content"] for chunk in sliding_window_chunks]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in sliding window: {len(duplicates)}")

contents = [chunk["content"] for chunk in processed_docs]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in fixed: {len(duplicates)}")

print(len(header_aware_embeddings))
print(len(sliding_window_embeddings))
print(len(processed_docs))


Duplicate chunks in header aware: 1
Duplicate chunks in sliding window: 0
Duplicate chunks in fixed: 0
82
69
51


### query rewrite

In [117]:
from ollama import Client

client = Client(host="http://127.0.0.1:11434")
def rewrite_query(query):
    system_prompt = (
            "You are an expert search engine query rewriter. Your job is to take a short, "
            "ambiguous user query and expand it into a dense, descriptive paragraph. "
            "Include technical synonyms, underlying concepts, and relevant terms. "
            "Do not answer the query. Only return the expanded search terms."
        )
    response = client.generate(
            model='qwen3:4b',
            prompt=f"System: {system_prompt}\nUser Query: {query}\nExpanded Query:"
        )    
    return response['response'].strip()

def retrieve_with_rewriter(query, embedding_matrix, model, chunks, k=5):
    expanded_query = rewrite_query(query)
    print(f"Expanded Query: {expanded_query}")
    return retrieve_top_chunks(expanded_query, embedding_matrix, model, chunks, k)

In [118]:
print("Retrieving top chunks for 'why do mini-batches work well on GPUs?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", header_aware_embeddings, model, header_aware_chunks, k=5))

Retrieving top chunks for 'why do mini-batches work well on GPUs?':


using fixed size chunking embeddings:
Expanded Query: The technical mechanisms that explain the effectiveness of mini-batch processing in deep learning training on graphics processing units, including GPU parallelism architecture, memory bandwidth constraints, gradient variance reduction through mini-batching, numerical stability improvements, optimal mini-batch size selection trade-offs, data parallelism, and hardware-level characteristics such as shared memory bandwidth, register file capacity, and compute core parallelism.
[{'chunk_index': np.int64(24), 'similarity_score': 0.8675747513771057, 'section': 'Neural Networks in Modern Deep Learning > Mini-Batch Gradient Descent', 'chunk_content': 'commonly uses mini-batches because they efficiently utilize parallel computation resources.\n\nThe term "parallel computation resources" will become important when discussing modern accelerators.\n\n---\n\n## Hardware Acceler

In [116]:
import os
import ollama

# Force python to ignore proxies for local loops
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# Or initialize an explicit local client bypassing environment configs
from ollama import Client
client = Client(host="http://127.0.0.1:11434")

# Test connection via tags endpoint
try:
    print("Available Local Models:", client.list())
except Exception as e:
    print("Still failed:", e)

Available Local Models: models=[Model(model='qwen3:4B', modified_at=datetime.datetime(2026, 6, 18, 10, 45, 42, 20448, tzinfo=TzInfo(19800)), digest='359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7', size=2497293931, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='4.0B', quantization_level='Q4_K_M')), Model(model='hermes3:latest', modified_at=datetime.datetime(2026, 6, 8, 2, 15, 32, 24685, tzinfo=TzInfo(19800)), digest='4f6b83f30b62bc3d0cf9be09266db222805ee815c8fd7d8b38f863f655be78b7', size=4661227243, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_0')), Model(model='qwen3:latest', modified_at=datetime.datetime(2026, 6, 8, 0, 11, 22, 174298, tzinfo=TzInfo(19800)), digest='500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41', size=5225388164, details=ModelDetails(parent_model='', format='gguf', family='qwen3', famil